In [ ]:
import json
import re
from pathlib import Path
from typing import List, Optional, Tuple

from docx import Document
from docx.text.paragraph import Paragraph
from docx.shared import RGBColor
from collections import defaultdict


# =========================
# CONFIG
# =========================
DATA_DIR = Path("infra/CUAD_v1/full_contract_docx")
contra_dir = Path("infra/docs_contradictions")
OUTPUT_DIR = Path("infra/CUAD_v1/full_contract_contradictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RED = RGBColor(255, 0, 0)


# =========================
# NORMALIZAÇÃO
# =========================
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def sentence_normalize(text: str) -> str:
    text = normalize_text(text)
    text = re.sub(r"\s*([,.;:!?])\s*", r"\1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# =========================
# DOCX HELPERS
# =========================
def iter_paragraphs(doc: Document) -> List[Paragraph]:
    """
    Pega parágrafos do corpo e de tabelas.
    """
    paragraphs = list(doc.paragraphs)

    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                paragraphs.extend(cell.paragraphs)

    return paragraphs


def clear_paragraph_runs(paragraph: Paragraph) -> None:
    """
    Remove runs, preservando propriedades do parágrafo.
    """
    p = paragraph._p
    for child in list(p):
        # remove tudo que não for pPr
        if child.tag.endswith("}pPr"):
            continue
        p.remove(child)


def copy_basic_run_style(src_run, dst_run):
    """
    Copia estilo básico de um run de referência.
    """
    if src_run is None:
        return

    try:
        dst_run.bold = src_run.bold
        dst_run.italic = src_run.italic
        dst_run.underline = src_run.underline
        dst_run.style = src_run.style
        if src_run.font is not None:
            dst_run.font.name = src_run.font.name
            dst_run.font.size = src_run.font.size
    except Exception:
        pass


def get_reference_run(paragraph: Paragraph):
    for run in paragraph.runs:
        if run.text and run.text.strip():
            return run
    return paragraph.runs[0] if paragraph.runs else None


def rewrite_paragraph_with_highlight(
    paragraph: Paragraph,
    before_text: str,
    highlight_text: str,
    after_text: str,
):
    """
    Reescreve o parágrafo preservando o estilo do parágrafo
    e usando vermelho apenas no trecho de contradição.
    """
    ref_run = get_reference_run(paragraph)

    clear_paragraph_runs(paragraph)

    if before_text:
        r1 = paragraph.add_run(before_text)
        copy_basic_run_style(ref_run, r1)

    if highlight_text:
        r2 = paragraph.add_run(highlight_text)
        copy_basic_run_style(ref_run, r2)
        r2.font.color.rgb = RED

    if after_text:
        r3 = paragraph.add_run(after_text)
        copy_basic_run_style(ref_run, r3)


def append_red_text(paragraph: Paragraph, text: str, add_newline: bool = True):
    ref_run = get_reference_run(paragraph)

    if add_newline and paragraph.runs:
        r_break = paragraph.add_run("\n")
        copy_basic_run_style(ref_run, r_break)

    r = paragraph.add_run(text)
    copy_basic_run_style(ref_run, r)
    r.font.color.rgb = RED


# =========================
# MATCHING
# =========================
def find_best_paragraph_for_sentence(
    paragraphs: List[Paragraph],
    target_sentence: str,
) -> Optional[Paragraph]:
    """
    Tenta achar o parágrafo que contém a sentença original.
    """
    target_norm = sentence_normalize(target_sentence)
    if not target_norm:
        return None

    # 1) match por containment exato após normalização
    for p in paragraphs:
        p_norm = sentence_normalize(p.text)
        if target_norm in p_norm:
            return p

    # 2) match flexível removendo excesso de espaços/pontuação
    target_compact = re.sub(r"[\s]+", " ", target_norm).strip()
    for p in paragraphs:
        p_norm = sentence_normalize(p.text)
        if target_compact in p_norm:
            return p

    return None


def find_best_paragraph_for_full_item(
    paragraphs: List[Paragraph],
    item_text: str,
) -> Optional[Paragraph]:
    """
    Fallback quando não achou pela sentença original.
    """
    item_norm = sentence_normalize(item_text)
    if not item_norm:
        return None

    # 1) igualdade total
    for p in paragraphs:
        if sentence_normalize(p.text) == item_norm:
            return p

    # 2) parágrafo maior que contém boa parte do texto
    for p in paragraphs:
        p_norm = sentence_normalize(p.text)
        if item_norm[:200] and item_norm[:200] in p_norm:
            return p

    return None


def split_original_and_contradiction_in_item(
    full_item_text: str,
    contradiction_text: str,
) -> Tuple[str, str]:
    """
    Quando o campo item['text'] já veio com a contradição embutida,
    tenta separar a parte original da parte contraditória.
    """
    full_norm = normalize_text(full_item_text)
    contr_norm = normalize_text(contradiction_text)

    idx = full_norm.find(contr_norm)
    if idx == -1:
        return full_norm, contr_norm

    before = full_norm[:idx].rstrip()
    return before, contr_norm


# =========================
# PROCESSAMENTO
# =========================
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_contradiction_items(data: list) -> list:
    items = []
    for item in data:
        if not isinstance(item, dict):
            continue
        if item.get("contradiction_replaced") and item.get("contradiction_metadata"):
            meta = item["contradiction_metadata"]
            if meta.get("llm_contradiction"):
                items.append(item)
    return items


def apply_contradictions_to_docx(docx_path: Path, json_path: Path, out_path: Path):
    data = load_json(json_path)
    contradiction_items = get_contradiction_items(data)

    if not contradiction_items:
        print(f"[SEM CONTRADIÇÕES] {json_path.name}")
        return

    doc = Document(str(docx_path))
    paragraphs = iter_paragraphs(doc)

    applied = 0
    not_found = []

    for item in contradiction_items:
        meta = item["contradiction_metadata"]
        mode = (meta.get("mode") or "insert").strip().lower()
        original_sentence = normalize_text(meta.get("original_sentence", ""))
        contradiction_text = normalize_text(meta.get("llm_contradiction", ""))
        full_item_text = normalize_text(item.get("text", ""))

        if not contradiction_text:
            continue

        target_p = None

        # primeiro tenta achar pelo original_sentence
        if original_sentence:
            target_p = find_best_paragraph_for_sentence(paragraphs, original_sentence)

        # fallback
        if target_p is None:
            target_p = find_best_paragraph_for_full_item(paragraphs, full_item_text)

        if target_p is None:
            not_found.append({
                "idx": item.get("idx"),
                "mode": mode,
                "original_sentence": original_sentence[:200],
                "contradiction": contradiction_text[:200],
            })
            continue

        p_text_norm = sentence_normalize(target_p.text)

        # =====================
        # MODE = REPLACE
        # =====================
        if mode == "replace" and original_sentence:
            orig_norm = sentence_normalize(original_sentence)

            if orig_norm in p_text_norm:
                # para conseguir preservar alguma estrutura, usa o texto já normalizado
                full_p_text = sentence_normalize(target_p.text)
                pos = full_p_text.find(orig_norm)

                before = full_p_text[:pos]
                after = full_p_text[pos + len(orig_norm):]

                rewrite_paragraph_with_highlight(
                    target_p,
                    before_text=before,
                    highlight_text=contradiction_text,
                    after_text=after,
                )
                applied += 1
                continue

            # fallback: se não conseguiu localizar direito, insere abaixo em vermelho
            append_red_text(target_p, contradiction_text, add_newline=True)
            applied += 1
            continue

        # =====================
        # MODE = INSERT
        # =====================
        if mode == "insert":
            # evita duplicar se por acaso já existir
            if sentence_normalize(contradiction_text) in sentence_normalize(target_p.text):
                continue

            append_red_text(target_p, contradiction_text, add_newline=True)
            applied += 1
            continue

        # =====================
        # fallback geral
        # =====================
        append_red_text(target_p, contradiction_text, add_newline=True)
        applied += 1

    doc.save(str(out_path))

    print(f"[OK] {docx_path.name} -> {out_path.name} | aplicadas={applied} | nao_encontradas={len(not_found)}")

    if not_found:
        log_path = out_path.with_suffix(".not_found.json")
        with open(log_path, "w", encoding="utf-8") as f:
            json.dump(not_found, f, ensure_ascii=False, indent=2)
        print(f"     log de falhas: {log_path.name}")


# =========================
# PAREAMENTO DOS ARQUIVOS
# =========================


def build_docx_index(data_dir: Path):
    """
    Cria um índice de todos os docx na estrutura:
    data_dir/Part_*/categoria/*.docx
    """
    docx_map = defaultdict(list)

    if not data_dir.exists():
        return docx_map

    for part_dir in data_dir.iterdir():
        if not part_dir.is_dir():
            continue

        if not part_dir.name.startswith("Part_"):
            continue

        for category_dir in part_dir.iterdir():
            if not category_dir.is_dir():
                continue

            for docx_path in category_dir.glob("*.docx"):
                docx_map[docx_path.stem].append(docx_path)

    return docx_map


def pair_files(data_dir: Path, contra_dir: Path):
    json_files = list(contra_dir.rglob("*.json"))
    docx_map = build_docx_index(data_dir)

    pairs = []
    for j in json_files:
        matches = docx_map.get(j.stem, [])

        if len(matches) == 1:
            pairs.append((j, matches[0]))
        elif len(matches) > 1:
            print(f"[MÚLTIPLOS DOCX] {j} -> usando {matches[0]}")
            pairs.append((j, matches[0]))
        else:
            print(f"[SEM DOCX CORRESPONDENTE] {j}")

    return pairs


def main():
    pairs = pair_files(DATA_DIR, contra_dir)

    if not pairs:
        print("Nenhum par JSON/DOCX encontrado.")
        return

    for json_path, docx_path in pairs:
        out_name = f"{docx_path.stem}_with_contradictions.docx"
        out_path = OUTPUT_DIR / out_name
        apply_contradictions_to_docx(docx_path, json_path, out_path)


if __name__ == "__main__":
    main()

[OK] ArcaUsTreasuryFund_20200207_N-2_EX-99.K5_11971930_EX-99.K5_Development Agreement.docx -> ArcaUsTreasuryFund_20200207_N-2_EX-99.K5_11971930_EX-99.K5_Development Agreement_with_contradictions.docx | aplicadas=2 | nao_encontradas=0
[OK] BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.docx -> BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement_with_contradictions.docx | aplicadas=2 | nao_encontradas=3
     log de falhas: BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement_with_contradictions.not_found.json
[OK] DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.docx -> DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement_with_contradictions.docx | aplicadas=4 | nao_encontradas=0
[OK] MusclepharmCorp_20170208_10-KA_EX-10.38_9893581_EX-10.38_Co-Branding Agreement.docx -> MusclepharmCorp_20170208_10-KA_EX-10.38_9893581_EX-10.38_Co-Branding Agreement_with_contradictions.docx |